<a href="https://colab.research.google.com/github/ajzal4you/Master-Project-/blob/main/pix2pix_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pix2Pix Inference: Generate Synthetic Masks from CT Images
Author: Ajzal

In [ ]:
!pip install torchvision

import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import os
from google.colab import files


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
# Define the generator (U-Net like)
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),
            nn.ReLU()
        )
        self.down2 = nn.Sequential(
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        u1 = self.up1(d2)
        u2 = self.up2(torch.cat([u1, d1], 1))
        return u2


In [ ]:
uploaded = files.upload()  # Select pix2pix_generator_epoch_50.pth

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = Generator().to(device)

model_path = list(uploaded.keys())[0]
generator.load_state_dict(torch.load(model_path, map_location=device))
generator.eval()

print(f"✅ Model loaded from {model_path}")


Saving pix2pix_generator_epoch_50.pth to pix2pix_generator_epoch_50.pth
✅ Model loaded from pix2pix_generator_epoch_50.pth


In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [ ]:
val_dir = "/content/kidney_pix2pix/val"
output_dir = "/content/generated_masks"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
import os

print(os.path.exists("/content/kidney_pix2pix/val"))  # True or False

False


In [ ]:
!ls /content


generated_masks  kidney_pix2pix  pix2pix_generator_epoch_50.pth  sample_data


In [ ]:
output_dir = "/content/kidney_pix2pix/val"
os.makedirs(output_dir, exist_ok=True)
# save inference results here

In [ ]:
!unzip -l /content/kidney_val.zip | head -n 40

from google.colab import drive; drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MSc_Project/GAN_Enhancement/exports"
!cp /content/kidney_val.zip "/content/drive/MyDrive/MSc_Project/GAN_Enhancement/exports/"

warning [/content/kidney_val.zip]:  zipfile is empty
Archive:  /content/kidney_val.zip
Mounted at /content/drive


In [ ]:
import os, cv2, numpy as np
from glob import glob

mask_dir = "/content/kidney_pix2pix/val"        # or "/content/generated_masks"
out_dir  = mask_dir.rstrip('/') + "_pp"
os.makedirs(out_dir, exist_ok=True)

for p in glob(os.path.join(mask_dir, "*")):
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if img is None: continue
    _, m = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    kernel = np.ones((3,3), np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN,  kernel, iterations=1)  # de-noise
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel, iterations=2)  # close gaps
    m = cv2.medianBlur(m, 3)                                        # smooth
    cv2.imwrite(os.path.join(out_dir, os.path.basename(p)), m)

print("Post-processed ->", out_dir)

Post-processed -> /content/kidney_pix2pix/val_pp


In [ ]:
# Use generated masks as predictions
pred_dir = "/content/generated_masks"
!find "$pred_dir" -maxdepth 1 -type f | wc -l


0


In [ ]:
import os, cv2, numpy as np, pandas as pd
from glob import glob

rows=[]
for p in glob(os.path.join(pred_dir, "*")):
    M=cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if M is None: continue
    rows.append({"file": os.path.basename(p), "mask_pixels": int((M>0).sum())})

df=pd.DataFrame(rows)
print(df.head())
df.to_csv("/content/gan_mask_areas.csv", index=False)
print("Saved -> /content/gan_mask_areas.csv")


Empty DataFrame
Columns: []
Index: []
Saved -> /content/gan_mask_areas.csv


In [ ]:
# Unzip back into val_pp
!rm -rf /content/kidney_pix2pix/val_pp
!mkdir -p /content/kidney_pix2pix/val_pp
!unzip -q /content/kidney_val_pp.zip -d /content/kidney_pix2pix/val_pp

# Verify files are there now
!find /content/kidney_pix2pix/val_pp -maxdepth 1 -type f | wc -l

warning [/content/kidney_val_pp.zip]:  zipfile is empty
0


In [ ]:
# before saving each output mask:
output_dir = "/content/kidney_pix2pix/val"   # or val_pp
import os; os.makedirs(output_dir, exist_ok=True)

# when saving each mask:
# cv2.imwrite(os.path.join(output_dir, filename), mask_img)


In [ ]:
# post-process from val -> val_pp (optional)
# ... your earlier post-processing code ...

# package
import shutil
shutil.make_archive("/content/kidney_val_pp", "zip", "/content/kidney_pix2pix/val_pp")

'/content/kidney_val_pp.zip'

In [ ]:
import shutil
final_dir = "/content/kidney_pix2pix/val_pp"   # post-processed folder
shutil.make_archive("/content/kidney_val_pp", 'zip', final_dir)
from google.colab import files; files.download("/content/kidney_val_pp.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
code = r'''
import torch, cv2, numpy as np, os
from torchvision import transforms

def load_pix2pix_generator(ckpt_path, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    from generator_model import Generator  # import your class
    G = Generator().to(device)
    G.load_state_dict(torch.load(ckpt_path, map_location=device))
    G.eval()
    return G, device

def run_pix2pix(G, device, img_path, out_path):
    img = cv2.imread(img_path)
    x = transforms.ToTensor()(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
    with torch.no_grad(): y = G(x)[0].cpu().numpy().transpose(1,2,0)
    m = (y*255).astype(np.uint8).squeeze()
    _, m = cv2.threshold(m, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    cv2.imwrite(out_path, m)
'''
with open("/content/pix2pix_infer_util.py","w") as f: f.write(code)
print("Saved /content/pix2pix_infer_util.py")


Saved /content/pix2pix_infer_util.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

val_dir = "/content/drive/MyDrive/kidney_pix2pix/val"

In [ ]:
for img_name in os.listdir(val_dir):
    img_path = os.path.join(val_dir, img_name)
    combined_img = Image.open(img_path).convert('RGB')

    # Crop left half (original CT image)
    left_half = combined_img.crop((0, 0, 256, 256))

    input_tensor = transform(left_half).unsqueeze(0).to(device)

    with torch.no_grad():
        fake_mask = generator(input_tensor)

    fake_mask_img = transforms.ToPILImage()(fake_mask.squeeze().cpu())
    fake_mask_img.save(f"{output_dir}/{img_name}")

    # Show side-by-side
    fig, axs = plt.subplots(1, 2, figsize=(8, 4))
    axs[0].imshow(left_half)
    axs[0].set_title("Original CT Image")
    axs[0].axis("off")
    axs[1].imshow(fake_mask_img, cmap='gray')
    axs[1].set_title("Generated Mask")
    axs[1].axis("off")
    plt.show()

print(f"✅ Masks saved in {output_dir}")


In [ ]:
import zipfile

zip_path = "/content/generated_masks.zip"
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, _, files_in_dir in os.walk(output_dir):
        for file in files_in_dir:
            zipf.write(os.path.join(root, file), file)

files.download(zip_path)
